# Virtual Multi-View Thermal Tracking Simulation

This notebook runs the full simulation pipeline: eagle motion → thermal rendering → voxel fusion → tracking → evaluation.

**To run on Google Colab:** Click `Runtime > Run all` or run cells one by one with `Shift+Enter`.

## 1. Install Dependencies

In [ ]:
print("Installing thermal_tracker from GitHub (may take 30-60s)...")
# Use --no-deps to avoid upgrading Colab's numpy/scipy to incompatible versions.
# Colab already has numpy, scipy, scikit-learn, matplotlib, pyyaml pre-installed.
!pip install -q --no-deps git+https://github.com/AliTranscrypts/wave.git@claude/start-app-implementation-oOMaO
!pip install -q pydantic opencv-python-headless
print("Done! Dependencies installed.")

# Verify multi-eagle support is present
from thermal_tracker.config import SimulationConfig
assert 'eagles' in SimulationConfig.model_fields, \
    "ERROR: 'eagles' field not found — old version. Restart runtime and re-run this cell."
print("Verified: multi-eagle support is active.")

## 2. Download Default Config

In [ ]:
import yaml
import math
import os

# Generate 12 cameras: 8 in outer ring + 4 in inner ring
cameras = []
for i in range(8):
    angle = 2.0 * math.pi * i / 8
    cameras.append({
        "id": f"cam_{i:02d}",
        "position": [round(500*math.cos(angle),1), round(500*math.sin(angle),1), 10.0],
        "orientation_mode": "look_at",
        "look_at_target": [0.0, 0.0, 100.0],
        "intrinsics_preset": "flir_640x512",
        "hfov_deg": 45.0,
    })
for i in range(4):
    angle = 2.0 * math.pi * i / 4 + math.radians(22.5)
    cameras.append({
        "id": f"cam_{8+i:02d}",
        "position": [round(300*math.cos(angle),1), round(300*math.sin(angle),1), 10.0],
        "orientation_mode": "look_at",
        "look_at_target": [0.0, 0.0, 150.0],
        "intrinsics_preset": "flir_640x512",
        "hfov_deg": 45.0,
    })

# 3 eagles with different Lissajous trajectories
# Each has different amplitudes, frequencies, phases → different flight paths
eagles = [
    {   # Eagle 1: wide figure-8 in the X-Y plane
        "temperature": 35.0, "radius": 0.5, "max_speed": 20.0, "max_acceleration": 5.0,
        "motion_type": "lissajous", "altitude_range": [50.0, 200.0],
        "lissajous_amplitudes": [280.0, 250.0, 25.0],
        "lissajous_frequencies": [0.05, 0.07, 0.03],
        "lissajous_phases": [0.0, 1.57, 0.0],
        "lissajous_z_center": 100.0,
        "initial_position": [0.0, 0.0, 100.0],
    },
    {   # Eagle 2: different phase → different part of the sky, slightly lower altitude
        "temperature": 36.0, "radius": 0.5, "max_speed": 18.0, "max_acceleration": 4.0,
        "motion_type": "lissajous", "altitude_range": [50.0, 200.0],
        "lissajous_amplitudes": [200.0, 300.0, 20.0],
        "lissajous_frequencies": [0.08, 0.04, 0.05],
        "lissajous_phases": [1.0, 0.0, 0.5],
        "lissajous_z_center": 90.0,
        "initial_position": [100.0, -50.0, 90.0],
    },
    {   # Eagle 3: tighter loop, higher altitude
        "temperature": 34.0, "radius": 0.6, "max_speed": 15.0, "max_acceleration": 3.0,
        "motion_type": "lissajous", "altitude_range": [50.0, 200.0],
        "lissajous_amplitudes": [150.0, 150.0, 30.0],
        "lissajous_frequencies": [0.1, 0.06, 0.04],
        "lissajous_phases": [2.0, 0.5, 1.0],
        "lissajous_z_center": 115.0,
        "initial_position": [-80.0, 100.0, 115.0],
    },
]

default_config = {
    "world": {
        "origin": [0.0, 0.0, 0.0],
        "x_min": -2500.0, "x_max": 2500.0,
        "y_min": -2500.0, "y_max": 2500.0,
        "z_min": 0.0, "z_max": 500.0
    },
    "eagle": eagles[0],         # Primary eagle (backward compat)
    "eagles": eagles,           # All 3 eagles
    "cameras": cameras,
    "rendering": {
        "sky_temperature": -10.0, "ground_temperature": 15.0,
        "atmospheric_attenuation_coeff": 0.0005,
        "vignetting_strength": 0.0, "blur_sigma_pixels": 2.0,
        "rendering_method": "projection",
        "noise": {"enabled": False, "gaussian_std": 0.05, "shot_noise_enabled": False,
                  "fixed_pattern_noise_std": 0.0, "random_seed": 42}
    },
    "run": {"num_frames": 1200, "dt": 0.1, "random_seed": 42}
}

os.makedirs("configs", exist_ok=True)
with open("configs/default_config.yaml", "w") as f:
    yaml.dump(default_config, f, default_flow_style=False)

print(f"Config saved: {len(eagles)} eagles, {len(cameras)} cameras, "
      f"{default_config['run']['num_frames']} frames ({default_config['run']['num_frames']*0.1:.0f}s)")
for i, e in enumerate(eagles):
    print(f"  Eagle {i+1}: start={e['initial_position']}, temp={e['temperature']}°C, "
          f"amp={e['lissajous_amplitudes']}, z_center={e['lissajous_z_center']}m")

## 3. Run the Simulation Engine

In [ ]:
import time
from thermal_tracker.config import load_config
from thermal_tracker.engine import SimulationEngine, SimulationOutputMode

config = load_config("configs/default_config.yaml")
n_cams = len(config.cameras)
n_eagles = len(config.eagles) if config.eagles else 1
mem_per_frame_mb = n_cams * 640 * 512 * 4 / 1e6
print(f"Config: {config.run.num_frames} frames ({config.run.num_frames * config.run.dt:.0f}s), "
      f"{n_eagles} eagles, {n_cams} cameras")
print(f"Using 50-frame batch for visualization.\n")

config_viz = load_config("configs/default_config.yaml")
config_viz.run.num_frames = 50

print(f"Rendering 50 visualization frames ({n_eagles} eagles x {n_cams} cameras)...")
t0 = time.time()
engine_viz = SimulationEngine(config_viz)
result = engine_viz.run(mode=SimulationOutputMode.BATCH)
print(f"  Done in {time.time()-t0:.1f}s | {engine_viz.num_eagles} eagles active")

# Compute all 3 full trajectories
print(f"\nComputing full {config.run.num_frames}-frame trajectories for {n_eagles} eagles...")
from thermal_tracker.eagle import create_motion_generator
import numpy as np

eagle_configs = config.eagles if config.eagles else [config.eagle]
all_trajs = []
for i, ecfg in enumerate(eagle_configs):
    gen = create_motion_generator(ecfg, config.world)
    traj = gen.generate_trajectory(config.run.num_frames, config.run.dt, seed=42 + i * 1000)
    all_trajs.append(traj)
    print(f"  Eagle {i+1}: X[{traj[:,1].min():.0f},{traj[:,1].max():.0f}] "
          f"Y[{traj[:,2].min():.0f},{traj[:,2].max():.0f}] "
          f"Z[{traj[:,3].min():.0f},{traj[:,3].max():.0f}]")

full_traj = all_trajs[0]  # Primary for backward compat
print("Ready for visualization.")

## 4. Visualize the Eagle Trajectory (3D)

In [ ]:
print("Rendering 3D trajectory plot for all eagles...")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

colors = ['blue', 'red', 'green']
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

for i, traj in enumerate(all_trajs):
    ax.plot(traj[:, 1], traj[:, 2], traj[:, 3], color=colors[i], linewidth=1.5,
            label=f'Eagle {i+1}', alpha=0.8)
    ax.scatter(traj[0, 1], traj[0, 2], traj[0, 3], color=colors[i], s=80, marker='o')
    ax.scatter(traj[-1, 1], traj[-1, 2], traj[-1, 3], color=colors[i], s=80, marker='x')

for cam_cfg in config.cameras:
    pos = cam_cfg.position
    ax.scatter(pos[0], pos[1], pos[2], c='orange', s=80, marker='^', alpha=0.7)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title(f'3 Eagle Trajectories ({config.run.num_frames * config.run.dt:.0f}s) & {len(config.cameras)} Cameras')
ax.legend()
plt.tight_layout()
plt.show()
print("Done.")

## 5. Visualize Thermal Images

In [ ]:
print("Generating thermal image visualization...")
frame = result.frame_bundles[25]
cam_ids = list(frame.camera_images.keys())

show_ids = [cam_ids[i] for i in range(0, len(cam_ids), max(1, len(cam_ids)//4))][:4]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, cam_id in enumerate(show_ids):
    img = frame.camera_images[cam_id]
    im = axes[i].imshow(img, cmap='inferno', aspect='auto')
    axes[i].set_title(f"{cam_id} (frame 25)")
    if cam_id in frame.ground_truth_2d and frame.visibility.get(cam_id, False):
        u, v = frame.ground_truth_2d[cam_id]
        axes[i].plot(u, v, 'g+', markersize=15, markeredgewidth=2)
    plt.colorbar(im, ax=axes[i], label='Temp (°C)')

plt.suptitle(f"Eagle at {frame.ground_truth_3d.round(1)} m  |  "
             f"Visible in: {[k for k,v in frame.visibility.items() if v]}", fontsize=11)
plt.tight_layout()
plt.show()

# Camera visibility stats from the viz batch
print("Computing camera visibility stats...")
from collections import Counter
vis_counts = Counter()
for fb in result.frame_bundles:
    for cam_id, visible in fb.visibility.items():
        if visible:
            vis_counts[cam_id] += 1
print(f"\nCamera visibility (first {result.num_frames} frames):")
for cam_id in sorted(vis_counts.keys()):
    pct = vis_counts[cam_id] / len(result.frame_bundles) * 100
    bar = "█" * int(pct / 2)
    print(f"  {cam_id}: {vis_counts[cam_id]:3d}/{len(result.frame_bundles)} ({pct:4.1f}%) {bar}")
print("Done.")

## 6. Run Full Tracking Pipeline

In [ ]:
import time, gc
from thermal_tracker.clustering import ClusteringConfig
from thermal_tracker.fusion import FusionConfig
from thermal_tracker.tracking import TrackingConfig
from thermal_tracker.voxel_grid import VoxelGridConfig
from thermal_tracker.pipeline import TrackingPipeline, generate_report
from thermal_tracker.tracking import TrackState

del result
gc.collect()
print("Freed visualization batch from memory.")

engine = SimulationEngine(config)
print(f"Engine: {engine.num_eagles} eagles, {len(engine.cameras)} cameras")

VOXEL_SIZE = 7.0
ROI_MIN = [-350.0, -350.0, 50.0]
ROI_MAX = [350.0, 350.0, 150.0]

nx = int((ROI_MAX[0] - ROI_MIN[0]) / VOXEL_SIZE)
ny = int((ROI_MAX[1] - ROI_MIN[1]) / VOXEL_SIZE)
nz = int((ROI_MAX[2] - ROI_MIN[2]) / VOXEL_SIZE)
total_voxels = nx * ny * nz
print(f"ROI: {nx}x{ny}x{nz} = {total_voxels:,} voxels (at {VOXEL_SIZE}m)")

pipeline = TrackingPipeline(
    cameras=engine.cameras,
    voxel_config=VoxelGridConfig(
        voxel_size=VOXEL_SIZE, roi_min=ROI_MIN, roi_max=ROI_MAX,
        occupancy_threshold=0.7, temporal_decay_rate=0.05,
    ),
    fusion_config=FusionConfig(
        detection_threshold_sigma=2.0, min_cameras_for_update=2,
    ),
    clustering_config=ClusteringConfig(
        method="connected_components", min_cluster_size=1, max_cluster_size=100,
    ),
    tracking_config=TrackingConfig(
        max_association_distance=30.0, min_hits_to_confirm=2, max_frames_to_coast=5,
        dt=config.run.dt,
        bounds_min=[ROI_MIN[0]-50, ROI_MIN[1]-50, ROI_MIN[2]-20],
        bounds_max=[ROI_MAX[0]+50, ROI_MAX[1]+50, ROI_MAX[2]+20],
        max_velocity=30.0, innovation_gate=5.0, max_oob_frames=3,
    ),
)

print(f"\nTracking {engine.num_eagles} eagles with zone awareness (no identity tracking)")
print(f"Kalman: position clamped, velocity≤30m/s, innovation gate=5σ, OOB kill after 3")

N = config.run.num_frames
print(f"\n{'='*80}")
print(f"  Running {N} frames — {engine.num_eagles} eagles — STREAMING mode")
print(f"{'='*80}")

tracking_results = []
gt_positions = []         # Primary eagle GT (backward compat)
all_gt_positions = []     # All eagles GT per frame: list of list of (3,) arrays
t0 = time.time()
frame_times = []

for i, frame_bundle in enumerate(engine.run_streaming()):
    ft0 = time.time()
    result_tr = pipeline.process_frame(frame_bundle)
    ft1 = time.time()
    frame_times.append(ft1 - ft0)
    tracking_results.append(result_tr)
    gt_positions.append([
        frame_bundle.timestamp,
        *frame_bundle.ground_truth_3d,
        *frame_bundle.ground_truth_velocity,
    ])
    all_gt_positions.append([p.copy() for p in frame_bundle.all_eagle_positions])

    if i == 0 or (i + 1) % 50 == 0 or (i + 1) == N:
        elapsed = time.time() - t0
        avg_ms = sum(frame_times) / len(frame_times) * 1000
        remaining = (N - i - 1) * (elapsed / (i + 1))
        n_tracks = len([t for t in result_tr.active_tracks if t.state == TrackState.CONFIRMED])
        n_vox = result_tr.num_active_voxels

        print(f"  Frame {i+1:4d}/{N} | "
              f"{elapsed:6.1f}s elapsed | "
              f"ETA {remaining:5.0f}s | "
              f"{avg_ms:5.0f}ms avg | "
              f"voxels: {n_vox:>6,} | "
              f"confirmed tracks: {n_tracks}/{engine.num_eagles}")

gt = np.array(gt_positions)
total = time.time() - t0
print(f"\n{'='*80}")
print(f"  DONE! {N} frames in {total:.1f}s ({total/N*1000:.0f}ms avg/frame)")
print(f"{'='*80}")

## 7. Evaluate Tracking Performance

In [ ]:
print("Computing multi-eagle evaluation metrics (zone awareness)...")
report = generate_report(tracking_results, gt, distance_threshold=20.0,
                         all_gt_positions=all_gt_positions)

n_eagles = len(config.eagles) if config.eagles else 1
print("="*55)
print(f"  EVALUATION REPORT — {n_eagles} Eagles, Zone Awareness")
print("="*55)
print(f"Duration:            {config.run.num_frames * config.run.dt:.0f}s ({config.run.num_frames} frames)")
print(f"Eagles:              {n_eagles}")
print(f"Cameras:             {len(config.cameras)}")
print(f"Voxel size:          {VOXEL_SIZE}m ({total_voxels:,} voxels)")
print(f"---")
print(f"Detection rate:      {report.detection_rate:.1%}  (eagle-frames with a nearby track)")
print(f"Mean position error: {report.mean_error:.2f} m  (avg nearest-track distance)")
print(f"Median error:        {report.median_error:.2f} m")
print(f"RMSE:                {report.rmse:.2f} m")
print(f"95th percentile:     {report.percentile_95:.2f} m")
print(f"Max error:           {report.max_error:.2f} m")
print(f"---")
print(f"Precision:           {report.precision:.1%}  (tracks that are near a real eagle)")
print(f"Recall:              {report.recall:.1%}  (eagles that have a nearby track)")
print(f"F1 score:            {report.f1_score:.1%}")
print(f"False positive rate: {report.false_positive_rate:.1%}")
print(f"---")
print(f"First detection:     frame {report.frames_to_first_detection}")
print(f"Longest gap:         {report.longest_detection_gap} frames")
print(f"Avg time/frame:      {report.mean_processing_time_ms:.1f} ms")
print(f"Max time/frame:      {report.max_processing_time_ms:.1f} ms")
print("Done.")

## 8. Plot Tracking Error Over Time

In [ ]:
print("Plotting tracking error and processing time over 1200 frames...")
from thermal_tracker.pipeline import compute_position_error

errors = compute_position_error(tracking_results, gt)
timestamps = gt[:, 0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax1.plot(timestamps, errors, 'b-', linewidth=0.8)
ax1.axhline(y=20.0, color='r', linestyle='--', alpha=0.5, label='Threshold (20m)')
ax1.set_ylabel('Position Error (m)')
ax1.set_title(f'Tracking Error Over 2-Minute Flight ({len(config.cameras)} cameras, {VOXEL_SIZE}m voxels)')
ax1.legend()
ax1.grid(True, alpha=0.3)

times = [r.processing_time_ms for r in tracking_results]
ax2.plot(timestamps, times, 'steelblue', linewidth=0.5)
ax2.set_ylabel('Processing Time (ms)')
ax2.set_xlabel('Time (s)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Done.")

## 9. Compare Ground Truth vs Tracked Trajectory

In [ ]:
print("Extracting tracked positions...")
tracked_frames = []
tracked_positions = []
for r in tracking_results:
    confirmed = [t for t in r.active_tracks if t.state == TrackState.CONFIRMED]
    for t in confirmed:
        pos = t.predicted_position
        if pos is not None:
            tracked_frames.append(r.frame_index)
            tracked_positions.append(pos)

print(f"  Total track points: {len(tracked_positions)} across {len(tracking_results)} frames")

if tracked_positions:
    print("Rendering 3D comparison: all GT eagles vs all tracks...")
    tracked = np.array(tracked_positions)

    fig = plt.figure(figsize=(14, 9))
    ax = fig.add_subplot(111, projection='3d')

    # GT trajectories (solid lines)
    colors = ['blue', 'red', 'green']
    for i, traj in enumerate(all_trajs):
        ax.plot(traj[:, 1], traj[:, 2], traj[:, 3], color=colors[i],
                linewidth=1.5, alpha=0.4, label=f'GT Eagle {i+1}')

    # All tracked positions (scatter — no identity, just presence)
    ax.scatter(tracked[:, 0], tracked[:, 1], tracked[:, 2],
               c='black', s=2, alpha=0.3, label='Tracked positions')

    for cam_cfg in config.cameras:
        pos = cam_cfg.position
        ax.scatter(pos[0], pos[1], pos[2], c='orange', s=60, marker='^', alpha=0.5)

    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(f'Zone Awareness: {len(all_trajs)} Eagles GT vs Tracked Positions')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print("Done.")
else:
    print("No tracked positions available.")

## 10. Experiment: Try Different Parameters

Modify the cells below to experiment with different settings.

In [ ]:
print("Generating random walk trajectory (50 frames)...")
config2 = load_config("configs/default_config.yaml")
config2.eagle.motion_type = "random_walk"
config2.run.num_frames = 50

engine3 = SimulationEngine(config2)
result3 = engine3.run(mode=SimulationOutputMode.BATCH)
print(f"  Done in {result3.wall_clock_seconds:.1f}s")

traj3 = result3.trajectory
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(traj3[:, 1], traj3[:, 2], traj3[:, 3], 'r-', linewidth=1.5)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title('Random Walk Trajectory')
plt.tight_layout()
plt.show()
print("Done.")

In [ ]:
print("Rendering 10 noisy frames...")
config3 = load_config("configs/default_config.yaml")
config3.rendering.noise.enabled = True
config3.rendering.noise.gaussian_std = 0.1
config3.run.num_frames = 10

engine4 = SimulationEngine(config3)
result4 = engine4.run(mode=SimulationOutputMode.BATCH)
print(f"  Done in {result4.wall_clock_seconds:.1f}s")

frame = result4.frame_bundles[5]
cam_ids_noise = list(frame.camera_images.keys())
show_noise = [cam_ids_noise[i] for i in range(0, len(cam_ids_noise), max(1, len(cam_ids_noise)//3))][:3]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, cam_id in enumerate(show_noise):
    img = frame.camera_images[cam_id]
    im = axes[i].imshow(img, cmap='inferno', aspect='auto')
    axes[i].set_title(f"{cam_id} (with noise)")
    plt.colorbar(im, ax=axes[i], label='Temp (°C)')
plt.suptitle('Thermal Images with Sensor Noise (std=0.1°C)', fontsize=12)
plt.tight_layout()
plt.show()
print("Done.")